In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()

True

In [3]:
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

In [4]:
print("Host:", DB_HOST)
print("Port:", DB_PORT)
print("Database:", DB_NAME)
print("User:", DB_USER)
print("Password configured:", DB_PASSWORD is not None)

Host: localhost
Port: 5432
Database: bikeshare
User: globalrex
Password configured: True


In [5]:
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME
)

In [6]:
engine = create_engine(connection_url)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT version();")
    )
    print(result.scalar())

PostgreSQL 18.6 (Ubuntu 18.6-0ubuntu0.26.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 15.2.0-16ubuntu1) 15.2.0, 64-bit


In [7]:
with engine.connect() as connection:
    existing_records = connection.execute(
        text("SELECT COUNT(*) FROM staging.hourly_rentals_raw")
    ).scalar()

    print(existing_records)

0


In [11]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "hourly_transformed.parquet"
)

print(INPUT_PATH)

transformed_data = pd.read_parquet(INPUT_PATH)

transformed_data.sample(5)

/home/global-rex/Documents/bikeshare-intelligence


FileNotFoundError: /data/processed/hourly_transformed.parquet

In [ ]:
if existing_records != 0:
    raise ValueError(
        "Staging table is not empty"
        "Check existing records before loading"
    )
transformed_data.to